In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from scipy.stats import uniform, randint

In [2]:
# Datasets
heart_test = pd.read_csv('Data/heart_test.csv')
heart_train = pd.read_csv('Data/heart_train.csv')
diabetes_test = pd.read_csv('Data/diabetes_test.csv')
diabetes_train = pd.read_csv('Data/diabetes_train.csv')
cancer_test = pd.read_csv('Data/cancer_test.csv')
cancer_train = pd.read_csv('Data/cancer_train.csv')
alzheimer_test = pd.read_csv('Data/alzheimer_test.csv')
alzheimer_train = pd.read_csv('Data/alzheimer_train.csv')

datasets = {
    "heart": (heart_train, heart_test),
    "diabetes": (diabetes_train, diabetes_test),
    "cancer": (cancer_train, cancer_test),
    "alzheimer": (alzheimer_train, alzheimer_test)
}

In [3]:
from sklearn.model_selection import train_test_split

# Training sets for 25%, 50%, 75%
heart_train25, _ = train_test_split(heart_train, train_size=0.25, random_state=42, stratify=heart_train.iloc[:, -1])
heart_train50, _ = train_test_split(heart_train, train_size=0.50, random_state=42, stratify=heart_train.iloc[:, -1])
heart_train75, _ = train_test_split(heart_train, train_size=0.75, random_state=42, stratify=heart_train.iloc[:, -1])

diabetes_train25, _ = train_test_split(diabetes_train, train_size=0.25, random_state=42, stratify=diabetes_train.iloc[:, -1])
diabetes_train50, _ = train_test_split(diabetes_train, train_size=0.50, random_state=42, stratify=diabetes_train.iloc[:, -1])
diabetes_train75, _ = train_test_split(diabetes_train, train_size=0.75, random_state=42, stratify=diabetes_train.iloc[:, -1])

cancer_train25, _ = train_test_split(cancer_train, train_size=0.25, random_state=42, stratify=cancer_train.iloc[:, -1])
cancer_train50, _ = train_test_split(cancer_train, train_size=0.50, random_state=42, stratify=cancer_train.iloc[:, -1])
cancer_train75, _ = train_test_split(cancer_train, train_size=0.75, random_state=42, stratify=cancer_train.iloc[:, -1])

alzheimer_train25, _ = train_test_split(alzheimer_train, train_size=0.25, random_state=42, stratify=alzheimer_train.iloc[:, -1])
alzheimer_train50, _ = train_test_split(alzheimer_train, train_size=0.50, random_state=42, stratify=alzheimer_train.iloc[:, -1])
alzheimer_train75, _ = train_test_split(alzheimer_train, train_size=0.75, random_state=42, stratify=alzheimer_train.iloc[:, -1])


datasets25 = {
    "heart": (heart_train25, heart_test),
    "diabetes": (diabetes_train25, diabetes_test),
    "cancer": (cancer_train25, cancer_test),
    "alzheimer": (alzheimer_train25, alzheimer_test)
}

datasets50 = {
    "heart": (heart_train50, heart_test),
    "diabetes": (diabetes_train50, diabetes_test),
    "cancer": (cancer_train50, cancer_test),
    "alzheimer": (alzheimer_train50, alzheimer_test)
}

datasets75 = {
    "heart": (heart_train75, heart_test),
    "diabetes": (diabetes_train75, diabetes_test),
    "cancer": (cancer_train75, cancer_test),
    "alzheimer": (alzheimer_train75, alzheimer_test)
}


# 1. Uniform Random

In [4]:
# Grid of hyperparameters 
param_uniform = {
    'n_estimators': randint(1, 5000),                      
    'learning_rate': [2 ** x for x in np.linspace(-10, 1, 50)],  
    'subsample': np.linspace(0.1, 0.999, 20),               
    'max_depth': randint(1, 15),                            
    'min_child_weight': [2 ** x for x in np.linspace(0, 7, 30)], 
    'colsample_bytree': np.linspace(0.1, 0.999, 20),        
    'colsample_bylevel': np.linspace(0.1, 0.999, 20),       
    'reg_lambda': [2 ** x for x in np.linspace(-10, 10, 40)],    
    'reg_alpha': [2 ** x for x in np.linspace(-10, 10, 40)]      
}


In [5]:
all_results = []

for name, (train, test) in datasets75.items():
    print(f"Training: {name}")
    
    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]
    
    # XGBoost model
    xgb = XGBClassifier(
        random_state=42,
        eval_metric='auc'  
    )

    
    # Random Search
    random_search = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=param_uniform,
        n_iter=100,                    
        scoring='roc_auc',
        cv=5,                          
        random_state=42,
        n_jobs=-1
    )
    
    # Fit
    random_search.fit(X_train, y_train)
    
    # Result
    cv_results = pd.DataFrame(random_search.cv_results_)
    
    # Testing on test sets
    for i, params in enumerate(random_search.cv_results_['params']):
        # Training again on parameters from random_search.cv_results_ :(
        model = XGBClassifier(
            random_state=42,
            eval_metric='auc',
            **params
        )
        model.fit(X_train, y_train)
        
        y_proba = model.predict_proba(X_test)[:, 1]
        test_auc = roc_auc_score(y_test, y_proba)
        
        all_results.append({
            "dataset": name,
            "params": params,
            "cv_roc_auc": cv_results.loc[i, 'mean_test_score'],
            "test_roc_auc": test_auc
        })
    
#Results
results_df = pd.DataFrame(all_results)

Training: heart
Training: diabetes
Training: cancer
Training: alzheimer


In [6]:
#Summary
for dataset in datasets75.keys():
    dataset_results = results_df[results_df['dataset'] == dataset]
    best_idx = dataset_results['test_roc_auc'].idxmax()
    best_result = dataset_results.loc[best_idx]
    
    print(f"\n{dataset.upper()}:")
    print(f"  Best test AUC: {best_result['test_roc_auc']:.4f}")
    print(f"  CV AUC: {best_result['cv_roc_auc']:.4f}")
    print(f"  Parameters: {best_result['params']}")


HEART:
  Best test AUC: 0.8067
  CV AUC: 0.7662
  Parameters: {'colsample_bylevel': 0.4312105263157895, 'colsample_bytree': 0.1, 'learning_rate': 0.0033909808852633535, 'max_depth': 11, 'min_child_weight': 33.56685617033712, 'n_estimators': 2807, 'reg_alpha': 7.064134542013183, 'reg_lambda': 173.15226339294657, 'subsample': 0.9043684210526316}

DIABETES:
  Best test AUC: 0.8308
  CV AUC: 0.7976
  Parameters: {'colsample_bylevel': 0.9043684210526316, 'colsample_bytree': 0.762421052631579, 'learning_rate': 0.025635479875238686, 'max_depth': 7, 'min_child_weight': 1.182122307190268, 'n_estimators': 155, 'reg_alpha': 0.0013933954886848076, 'reg_lambda': 7.064134542013183, 'subsample': 0.8570526315789474}

CANCER:
  Best test AUC: 0.8610
  CV AUC: 0.8631
  Parameters: {'colsample_bylevel': 0.6204736842105263, 'colsample_bytree': 0.999, 'learning_rate': 0.08901572837528347, 'max_depth': 14, 'min_child_weight': 5.328735052946112, 'n_estimators': 1664, 'reg_alpha': 14.381615571128657, 'reg_la

In [7]:
#Finding the best set of hyperparameters for each dataset 
best_per_dataset = (
    results_df.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False]).groupby("dataset", as_index=False).first()
)
params_df = best_per_dataset["params"].apply(pd.Series)

# Creating new set of hyperparameters from all datasets
mean_params = params_df.mean()
mean_params_dict = mean_params.to_dict()

for param in ["max_depth", "min_child_weight", "n_estimators"]:
    mean_params_dict[param] = int(round(mean_params_dict[param]))
mean_results = []

#Training with new hyperparameters on all datasets
for name, (train, test) in datasets50.items():

    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    model = XGBClassifier(
            random_state=42,
            eval_metric='auc',
            **mean_params_dict
        )
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    mean_auc = roc_auc_score(y_test, y_proba)

    mean_results.append({
        "dataset": name,
        "star_test_roc_auc": mean_auc
    })


In [8]:
# Creating new dataframe with all results

# Star means that this set of hyperparameters is a mean from the best 4 sets of hyperparameters, one for each set
params_df = results_df['params'].apply(pd.Series)
results_df = pd.concat([results_df.drop('params', axis=1), params_df], axis=1)

results_col = results_df[['cv_roc_auc','test_roc_auc']]
results_df = pd.concat([results_df.drop(['cv_roc_auc','test_roc_auc'],axis=1),results_col], axis=1)

mean_df = pd.DataFrame(mean_results)
results_df = results_df.merge(mean_df, on='dataset')
results_df['diff_from_star'] = results_df['star_test_roc_auc'] - results_df['test_roc_auc']
#results_df

In [9]:
results_df.to_csv("Results/xgboost_uniform_75.csv", index=False)

In [10]:
# Best parameters for each dataset
best_per_dataset = (
    results_df.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False])
    .groupby("dataset", as_index=False)
    .first()
    .drop(['cv_roc_auc', 'diff_from_star'], axis=1)
    #.drop(['star_test_roc_auc', 'diff_from_star'], axis=1)
)

In [11]:
# Deafault model

default_results = []
for name, (train, test) in datasets75.items():

    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    
    model = XGBClassifier(random_state=42,)
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_proba)

    default_results.append({
        "dataset": name,
        "default_test_roc_auc": score
    })

default_df = pd.DataFrame(default_results)
summary_df = best_per_dataset.merge(default_df, on="dataset")

In [12]:
# STAR row
mean_row = {
    "dataset": "STAR",
    **mean_params_dict,
    # "ccp_alpha" :None, "max_depth": None, "min_samples_leaf": None, "min_samples_split": None,
    "test_roc_auc": None,
    "star_test_roc_auc": mean_df["star_test_roc_auc"].mean(),
    #"star_test_roc_auc": None,
    "default_test_roc_auc": None
}

summary_df = pd.concat([summary_df, pd.DataFrame([mean_row])], ignore_index=True)
summary_df

C:\Users\milek\AppData\Local\Temp\ipykernel_27040\167484079.py:12: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary_df = pd.concat([summary_df, pd.DataFrame([mean_row])], ignore_index=True)


,dataset,colsample_bylevel,colsample_bytree,learning_rate,max_depth,min_child_weight,n_estimators,reg_alpha,reg_lambda,subsample,test_roc_auc,star_test_roc_auc,default_test_roc_auc
0,alzheimer,0.857053,0.715105,0.001820,12.0,12.300880,2266.0,7.064135,1.194503,0.383895,0.861010,0.852858,0.844601
1,cancer,0.620474,0.999000,0.089016,14.0,5.328735,1664.0,14.381616,4.950905,0.383895,0.861015,0.852829,0.809989
2,diabetes,0.904368,0.762421,0.025635,7.0,1.182122,155.0,0.001393,7.064135,0.857053,0.830806,0.801313,0.797612
3,heart,0.431211,0.100000,0.003391,11.0,33.566856,2807.0,7.064135,173.152263,0.904368,0.806679,0.804042,0.765719
4,STAR,0.703276,0.644132,0.029965,11.0,13.000000,1723.0,7.127820,46.590451,0.632303,NaN,0.827761,NaN


In [13]:
summary_df.to_csv("Results/xgboost_uniform_summary_75.csv", index=False)

# 2. Bayesian

In [14]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from sklearn.metrics import roc_auc_score

In [15]:
search_spaces = {
    'n_estimators': Integer(1, 5000),                        
    'learning_rate': Real(2**-10, 2**1, prior='log-uniform'),   
    'subsample': Real(0.1, 1.0, prior='uniform'),                
    'max_depth': Integer(1, 15),                              
    'min_child_weight': Real(2**0, 2**7, prior='log-uniform'), 
    'colsample_bytree': Real(0.1, 1.0, prior='uniform'),       
    'colsample_bylevel': Real(0.1, 1.0, prior='uniform'),      
    'reg_lambda': Real(2**-10, 2**10, prior='log-uniform'),    
    'reg_alpha': Real(2**-10, 2**10, prior='log-uniform')      
}


In [16]:
all_results_2 = []

for name, (train, test) in datasets75.items():
    print(f"Training: {name}")
    
    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]
    
    # XGBoost model
    xgb = XGBClassifier(
        random_state=42,
        eval_metric='auc'
    )
    
    # Bayesian Search
    bayes_search = BayesSearchCV(
        estimator=xgb,
        search_spaces=search_spaces,
        n_iter=100,
        scoring='roc_auc',
        cv=5,
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    
    # Fit
    bayes_search.fit(X_train, y_train)
    
    # Result
    cv_results = pd.DataFrame(bayes_search.cv_results_)
 
    # Testing on test sets
    for i, params in enumerate(bayes_search.cv_results_['params']):
        # Training again on parameters from bayes_search.cv_results_
        model = XGBClassifier(
            random_state=42,
            eval_metric='auc',
            **params
        )
        model.fit(X_train, y_train)
        
        y_proba = model.predict_proba(X_test)[:, 1]
        test_auc = roc_auc_score(y_test, y_proba)
        
        all_results_2.append({
            "dataset": name,
            "params": params,
            "cv_roc_auc": cv_results.loc[i, 'mean_test_score'],
            "test_roc_auc": test_auc
        })

# Results
results_2_df = pd.DataFrame(all_results_2)

Training: heart
Training: diabetes
Training: cancer
Training: alzheimer


In [17]:
# Summary
for dataset in datasets75.keys():
    dataset_results = results_2_df[results_2_df['dataset'] == dataset]
    best_idx = dataset_results['test_roc_auc'].idxmax()
    best_result = dataset_results.loc[best_idx]
    
    print(f"\n{dataset.upper()}:")
    print(f"  Best test AUC: {best_result['test_roc_auc']:.4f}")
    print(f"  CV AUC: {best_result['cv_roc_auc']:.4f}")
    print(f"  Parameters: {best_result['params']}")



HEART:
  Best test AUC: 0.8068
  CV AUC: 0.7843
  Parameters: OrderedDict({'colsample_bylevel': 0.2511501529257252, 'colsample_bytree': 0.35114405973615714, 'learning_rate': 0.0029449154116711687, 'max_depth': 6, 'min_child_weight': 15.993335315304067, 'n_estimators': 775, 'reg_alpha': 12.236626442909838, 'reg_lambda': 377.4313996000459, 'subsample': 0.7429294979696429})

DIABETES:
  Best test AUC: 0.8352
  CV AUC: 0.7988
  Parameters: OrderedDict({'colsample_bylevel': 0.9429279111463462, 'colsample_bytree': 1.0, 'learning_rate': 0.0009765625, 'max_depth': 4, 'min_child_weight': 3.504679392814365, 'n_estimators': 1938, 'reg_alpha': 0.013217728923777192, 'reg_lambda': 0.0009765625, 'subsample': 0.8636924124824099})

CANCER:
  Best test AUC: 0.8607
  CV AUC: 0.8629
  Parameters: OrderedDict({'colsample_bylevel': 0.3639510792162838, 'colsample_bytree': 1.0, 'learning_rate': 0.0039699308585632595, 'max_depth': 9, 'min_child_weight': 1.0, 'n_estimators': 3629, 'reg_alpha': 0.00940990664167

In [18]:
# Finding the best hyperparameters for each dataset
best_per_dataset_2 = (
    results_2_df.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False]).groupby("dataset", as_index=False).first()
)

params_2_df = best_per_dataset_2["params"].apply(pd.Series)

In [19]:
# Creating a final dataframe for all of the results

# Splitting all hyperparaemters to separate columns
params_2_df = results_2_df['params'].apply(pd.Series)
results_2_df = pd.concat([results_2_df.drop('params', axis=1), params_2_df], axis=1)

results_col_2 = results_2_df[['cv_roc_auc','test_roc_auc']]
results_2_df = pd.concat([results_2_df.drop(['cv_roc_auc','test_roc_auc'],axis=1),results_col_2], axis=1)

In [20]:
results_2_df.to_csv("Results/xgboost_bayes_75.csv", index=False)

In [21]:
# Best parameters for each dataset
best_per_dataset_2 = (
    results_2_df.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False])
    .groupby("dataset", as_index=False)
    .first()
    .drop(['cv_roc_auc'], axis=1)
)

summary_2_df = best_per_dataset_2.merge(default_df, on="dataset")

In [22]:
summary_2_df.to_csv("Results/xgboost_bayes_summary_75.csv", index=False)